# Arm 2: Primary Pool (FAA) + Kuramoto

**Objective.** Test Decision 6's actual claim: does adding Kuramoto order parameter and metastability to the primary pool improve predictive performance over FAA alone? Same four classifiers (Ledoit-Wolf shrinkage LDA, elastic-net logistic regression, L2/"Bayesian" logistic regression, unregularized logistic regression), same nested-CV and permutation-testing procedure as Arm 1, so any difference reflects the added features, not a change in method.

This notebook imports `AgeDeconfounder` and `run_nested_cv` from `src/modelling.py` rather than redefining them - both were built and validated in Arm 1 (`08_arm1_primary_pool.ipynb`).

**Inputs.**
- `full_cohort_features.parquet`, restEC/heog_off, subset to alpha power (F3, F4, for FAA) and Kuramoto order parameter + metastability (all 5 bands)
- Responder/non-responder labels
- Age, for fold-scoped deconfounding

**Feature construction.** FAA as in Arm 1 (raw F4−F3, alpha power). Kuramoto: order parameter and metastability across all five bands (delta, theta, alpha, beta, gamma), global across all 26 channels (Decision 6) - 10 columns, not log-transformed (Decision 2, extended to Kuramoto: both metrics are bounded quantities, same reasoning as PLI/coherence/PLV, not right-skewed spectral power). Primary pool is FAA + all 10 Kuramoto columns, p=11 total.

**Assumptions.**
- Same fold-scoped age deconfounding, same four classifiers, same class-balanced weighting, same `StratifiedKFold` configuration (5 outer / 3 inner folds, `RANDOM_STATE=42`) as Arm 1 - held constant deliberately, so this arm is a clean comparison, not a re-tuned one.
- This is a test of whether synchrony *adds* value beyond FAA, not a claim that Kuramoto features predict response on their own (Decision 6).
- At p=11 (vs. Arm 1's p=1), the AUC-tie seen in Arm 1 is not expected to hold — different classifiers can now rank subjects differently. Worth checking directly rather than assuming.
- Balanced accuracy remains the primary metric; sensitivity, specificity, and PPV are also reported (Decision 5).

In [6]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from scipy import stats

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression

# Temporary bootstrap path, just to make src/ importable — not the real project root
sys.path.insert(0, str(Path.cwd().parent))
from src.modelling import AgeDeconfounder, run_nested_cv
from src.preprocessing import find_repo_root

In [7]:
## Data loading 

project_root = find_repo_root()
data_dir = project_root / "data"
features_dir = data_dir / "features"

full_df = pd.read_parquet(features_dir / "full_cohort_features.parquet")
alpha_power_columns = full_df[['F3_alpha_power', 'F4_alpha_power']]
print(alpha_power_columns)

# Confirm the real Kuramoto column names
print([c for c in full_df.columns if c.lower().startswith('delta_')])

     F3_alpha_power  F4_alpha_power
0        135.931109      141.647315
1         47.927017       52.177002
2         41.122609       40.342005
3        281.296923      308.598717
4        181.383911      170.625480
..              ...             ...
155      283.047167      262.343693
156       38.087251       38.843060
157      294.013122      321.013081
158      291.411640      323.286509
159       99.080413       96.545709

[160 rows x 2 columns]
['delta_order', 'delta_metastability']


In [8]:
#Feature construction: Kuramoto order parameter and metastability for each band
kuramoto_cols = [f"{band}_order" for band in ['delta', 'theta', 'alpha', 'beta', 'gamma']] + \
                [f"{band}_metastability" for band in ['delta', 'theta', 'alpha', 'beta', 'gamma']]

# Confirm all 10 actually exist before using them 
missing = set(kuramoto_cols) - set(full_df.columns)
print(len(kuramoto_cols), "expected")
print("missing:", missing)   # must be empty

kuramoto_df = full_df[kuramoto_cols]
kuramoto_df.head()

10 expected
missing: set()


,delta_order,theta_order,alpha_order,beta_order,gamma_order,delta_metastability,theta_metastability,alpha_metastability,beta_metastability,gamma_metastability
0,0.671842,0.659621,0.635246,0.597260,0.638309,0.204145,0.203220,0.204042,0.219166,0.225556
1,0.651563,0.683092,0.674080,0.729619,0.819653,0.197602,0.203627,0.204438,0.199507,0.177209
2,0.674987,0.727175,0.666043,0.561805,0.555825,0.191856,0.189211,0.198968,0.210846,0.208163
3,0.701484,0.708950,0.677826,0.640632,0.637159,0.200143,0.191657,0.197684,0.219111,0.218663
4,0.691333,0.697773,0.591881,0.614711,0.589367,0.197237,0.204834,0.200559,0.217889,0.220254


In [9]:
# Build FAA - same construction as Arm 1
faa = alpha_power_columns['F4_alpha_power'] - alpha_power_columns['F3_alpha_power']

# Merge FAA + Kuramoto with age and responder status, keyed on subject ID
model_df = pd.concat([
    full_df[['subject_id']],
    faa.rename('FAA'),
    kuramoto_df,
], axis=1)

cohort_df = pd.read_excel(data_dir / "cohort_filtered_n163.xlsx")

model_df = model_df.merge(
    cohort_df[['TDBRAIN_ID', 'age', 'Responder']],
    left_on='subject_id', right_on='TDBRAIN_ID', how='left'
).drop(columns='TDBRAIN_ID')

feature_cols = ['FAA'] + kuramoto_cols

print(len(model_df))                                              # must stay 160
print(model_df[feature_cols + ['age', 'Responder']].isna().sum()) # must all be 0
print(model_df['Responder'].value_counts())                        # expect ~93/67, matching Arm 1
model_df.head()

160
FAA                    0
delta_order            0
theta_order            0
alpha_order            0
beta_order             0
gamma_order            0
delta_metastability    0
theta_metastability    0
alpha_metastability    0
beta_metastability     0
gamma_metastability    0
age                    0
Responder              0
dtype: int64
Responder
1    93
0    67
Name: count, dtype: int64


,subject_id,FAA,delta_order,theta_order,alpha_order,beta_order,gamma_order,delta_metastability,theta_metastability,alpha_metastability,beta_metastability,gamma_metastability,age,Responder
0,sub-88045809,5.716206,0.671842,0.659621,0.635246,0.597260,0.638309,0.204145,0.203220,0.204042,0.219166,0.225556,43.06,0
1,sub-88022765,4.249985,0.651563,0.683092,0.674080,0.729619,0.819653,0.197602,0.203627,0.204438,0.199507,0.177209,32.96,1
2,sub-88023485,-0.780604,0.674987,0.727175,0.666043,0.561805,0.555825,0.191856,0.189211,0.198968,0.210846,0.208163,44.50,1
3,sub-88061061,27.301794,0.701484,0.708950,0.677826,0.640632,0.637159,0.200143,0.191657,0.197684,0.219111,0.218663,27.69,1
4,sub-88021321,-10.758431,0.691333,0.697773,0.591881,0.614711,0.589367,0.197237,0.204834,0.200559,0.217889,0.220254,28.28,0


In [12]:
# Nested CV: FAA + Kuramoto, class-balanced weighting (same config as Arm 1)

N_OUTER_SPLITS = 5
N_INNER_SPLITS = 3
RANDOM_STATE = 42

X = model_df[feature_cols].values
y = model_df['Responder'].values
age = model_df['age'].values

classifier_specs_balanced = {
    'LDA (Ledoit-Wolf, balanced priors)': (
        LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto', priors=[0.5, 0.5]),
        None
    ),
    'Logistic (unregularized, balanced)': (
        LogisticRegression(C=np.inf, max_iter=1000, class_weight='balanced'),
        None
    ),
    'Logistic (elastic-net, balanced)': (
        LogisticRegression(solver='saga', max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE),
        {'C': [0.01, 0.1, 1, 10, 100], 'l1_ratio': [0.1, 0.5, 0.9]}
    ),
    'Logistic (L2 / "Bayesian" MAP, balanced)': (
        LogisticRegression(l1_ratio=0, max_iter=1000, class_weight='balanced'),
        {'C': [0.01, 0.1, 1, 10, 100]}
    ),
}

results_df = run_nested_cv(X, y, age, classifier_specs_balanced, N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)

print(results_df.isna().sum())
print(results_df.groupby('classifier')[['balanced_accuracy', 'accuracy', 'auc', 'sensitivity', 'specificity', 'ppv']].mean())
results_df

classifier           0
fold                 0
balanced_accuracy    0
accuracy             0
auc                  0
sensitivity          0
specificity          0
ppv                  0
dtype: int64
                                          balanced_accuracy  accuracy  \
classifier                                                              
LDA (Ledoit-Wolf, balanced priors)                 0.597417   0.58750   
Logistic (L2 / "Bayesian" MAP, balanced)           0.618309   0.61250   
Logistic (elastic-net, balanced)                   0.613045   0.60625   
Logistic (unregularized, balanced)                 0.609077   0.60000   

                                               auc  sensitivity  specificity  \
classifier                                                                     
LDA (Ledoit-Wolf, balanced priors)        0.613531     0.550877     0.643956   
Logistic (L2 / "Bayesian" MAP, balanced)  0.635817     0.605848     0.630769   
Logistic (elastic-net, balanced)          0.

,classifier,fold,balanced_accuracy,accuracy,auc,sensitivity,specificity,ppv
0,"LDA (Ledoit-Wolf, balanced priors)",0,0.555556,0.56250,0.591270,0.611111,0.500000,0.611111
1,"Logistic (unregularized, balanced)",0,0.555556,0.56250,0.638889,0.611111,0.500000,0.611111
2,"Logistic (elastic-net, balanced)",0,0.611111,0.62500,0.607143,0.722222,0.500000,0.650000
3,"Logistic (L2 / ""Bayesian"" MAP, balanced)",0,0.611111,0.62500,0.619048,0.722222,0.500000,0.650000
4,"LDA (Ledoit-Wolf, balanced priors)",1,0.682540,0.68750,0.666667,0.722222,0.642857,0.722222
5,"Logistic (unregularized, balanced)",1,0.702381,0.71875,0.769841,0.833333,0.571429,0.714286
6,"Logistic (elastic-net, balanced)",1,0.666667,0.68750,0.738095,0.833333,0.500000,0.681818
7,"Logistic (L2 / ""Bayesian"" MAP, balanced)",1,0.666667,0.68750,0.734127,0.833333,0.500000,0.681818
8,"LDA (Ledoit-Wolf, balanced priors)",2,0.582996,0.56250,0.647773,0.473684,0.692308,0.692308
9,"Logistic (unregularized, balanced)",2,0.609312,0.59375,0.599190,0.526316,0.692308,0.714286


In [14]:
# Permutation test: is each classifier's mean balanced accuracy distinguishable from chance?

N_PERMUTATIONS = 1000
rng = np.random.RandomState(RANDOM_STATE)

# Observed statistic: reuse what's already computed, don't rerun
observed = results_df.groupby('classifier')['balanced_accuracy'].mean().to_dict()

# Null distribution: same procedure, shuffled labels
print(f"N_PERMUTATIONS = {N_PERMUTATIONS}")

null_scores = {name: [] for name in classifier_specs_balanced}

for i in range(N_PERMUTATIONS):
    y_shuffled = rng.permutation(y)
    perm_df = run_nested_cv(X, y_shuffled, age, classifier_specs_balanced, N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)
    perm_result = perm_df.groupby('classifier')['balanced_accuracy'].mean().to_dict()
    for name, score in perm_result.items():
        null_scores[name].append(score)

# p-value: fraction of the null distribution at or above the observed value
# (+1/+1 correction: the observed result is itself one valid draw under the null)
print(f"{'classifier':<45} {'observed':>10} {'null mean':>10} {'p-value':>10}")
for name in classifier_specs_balanced:
    null_arr = np.array(null_scores[name])
    p_value = (np.sum(null_arr >= observed[name]) + 1) / (N_PERMUTATIONS + 1)
    print(f"{name:<45} {observed[name]:>10.4f} {null_arr.mean():>10.4f} {p_value:>10.4f}")

N_PERMUTATIONS = 1000
classifier                                      observed  null mean    p-value
LDA (Ledoit-Wolf, balanced priors)                0.5974     0.5003     0.0250
Logistic (unregularized, balanced)                0.6091     0.5002     0.0120
Logistic (elastic-net, balanced)                  0.6130     0.5010     0.0100
Logistic (L2 / "Bayesian" MAP, balanced)          0.6183     0.5005     0.0080


### Direct comparison: does adding Kuramoto improve on FAA alone?

Arm 1 and Arm 2 each cleared their own permutation test, but that doesn't establish that Arm 2 is *better* than Arm 1 - two independently significant results can differ from each other by an amount that's well within noise. The correct test is a **paired permutation test on the difference in balanced accuracy**, not a comparison of two p-values.

Both arms already use the same `StratifiedKFold` split (same `RANDOM_STATE`), so fold-for-fold comparisons are valid - same subjects in each fold, only the feature set differs. The null distribution here is built the same way as before, but on the *difference*: for each of 1000 permutations, the same shuffled label set is used to compute balanced accuracy for both FAA alone and FAA+Kuramoto, and the difference between them is recorded. The observed difference (real labels) is then compared against that null-of-differences - this directly tests whether adding Kuramoto helps *more than chance variation alone would produce*, for each classifier.

This reruns nested CV for both feature sets, every permutation - expect this to take at least as long as Arm 2's own permutation test, plausibly longer.

In [15]:
# Paired permutation test: does FAA+Kuramoto beat FAA alone, per classifier?

X_faa_only = model_df[['FAA']].values  # Arm 1's feature set, already available here

# Observed difference: real labels, both feature sets
arm1_observed_df = run_nested_cv(X_faa_only, y, age, classifier_specs_balanced, N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)
arm1_observed = arm1_observed_df.groupby('classifier')['balanced_accuracy'].mean().to_dict()

observed_diff = {name: observed[name] - arm1_observed[name] for name in classifier_specs_balanced}

# Null distribution of differences: same shuffled y used for both feature sets,
# each permutation, so the comparison stays paired throughout
rng = np.random.RandomState(RANDOM_STATE)
null_diffs = {name: [] for name in classifier_specs_balanced}

for i in range(N_PERMUTATIONS):
    y_shuffled = rng.permutation(y)

    perm_arm2 = run_nested_cv(X, y_shuffled, age, classifier_specs_balanced, N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)
    perm_arm2_scores = perm_arm2.groupby('classifier')['balanced_accuracy'].mean().to_dict()

    perm_arm1 = run_nested_cv(X_faa_only, y_shuffled, age, classifier_specs_balanced, N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)
    perm_arm1_scores = perm_arm1.groupby('classifier')['balanced_accuracy'].mean().to_dict()

    for name in classifier_specs_balanced:
        null_diffs[name].append(perm_arm2_scores[name] - perm_arm1_scores[name])

# p-value: fraction of the null difference distribution at or above the observed difference
print(f"{'classifier':<45} {'obs diff':>10} {'null mean':>10} {'p-value':>10}")
for name in classifier_specs_balanced:
    null_arr = np.array(null_diffs[name])
    p_value = (np.sum(null_arr >= observed_diff[name]) + 1) / (N_PERMUTATIONS + 1)
    print(f"{name:<45} {observed_diff[name]:>10.4f} {null_arr.mean():>10.4f} {p_value:>10.4f}")

classifier                                      obs diff  null mean    p-value
LDA (Ledoit-Wolf, balanced priors)                0.0268     0.0029     0.3626
Logistic (unregularized, balanced)                0.0385     0.0027     0.2867
Logistic (elastic-net, balanced)                  0.0424     0.0022     0.2188
Logistic (L2 / "Bayesian" MAP, balanced)          0.0477     0.0029     0.2448


### Interpretation

Arm 2 (FAA + Kuramoto) clears conventional significance for all four classifiers (balanced accuracy 0.597–0.618, p = 0.008–0.025) - a stronger, more consistent result than Arm 1's single significant classifier. But the direct paired comparison against Arm 1 tells a different story: none of the four classifiers show a statistically significant improvement from adding Kuramoto (p = 0.22-0.36), despite every classifier's raw balanced accuracy being higher in Arm 2 than Arm 1.

These aren't contradictory. Arm 2's permutation test asks "does FAA+Kuramoto together beat chance?" - yes. The paired test asks "does adding Kuramoto beat FAA alone by more than chance would produce?" - not demonstrated at this sample size. The second question is the one Decision 6 was actually built to answer, and the honest result is that synchrony's added value isn't established here, not that it's been shown absent - a null result at n=160 with an 11-feature model is consistent with a real but small effect the sample can't resolve, not proof there's nothing there.

## Summary

**Findings.** FAA + Kuramoto predicts responder status above chance across all four classifiers (balanced accuracy 0.597–0.618, p = 0.008–0.025) - a more consistent result than Arm 1's FAA-alone finding. However, a direct paired permutation test found no significant improvement over FAA alone for any classifier (p = 0.22–0.36): Decision 6's core question - does synchrony add value beyond standard features - is not resolved by this arm, despite Arm 2 clearing its own significance test.

**Residual assumptions, carried into Arm 3 onward.**
- Fold-scoped `StandardScaler`, added to `run_nested_cv` in this notebook after unscaled features (FAA range ~134, Kuramoto range ~0.1–1) caused `saga` convergence failures - now a standing part of the shared harness, not specific to this arm.
- Arm 1's results were reconfirmed identical after this change (single-feature case is scale-invariant for the classifiers used here); worth re-checking this assumption if a future arm's feature set behaves unexpectedly.
- The paired-comparison pattern (same shuffled labels across both feature sets, same folds) is reusable for any future "does X add value over baseline" question - worth extracting into `src/modelling.py` if Arm 3 or later needs the same comparison.
- Grid search boundaries (`C`, `l1_ratio`) are unchanged from Arm 1 - still not independently validated for this feature count.